# Task 1: Model Performance Comparison

This notebook supports Task 1 of the TechTrack case analysis. It loads the full-dataset evaluation results for Model 1 and Model 2, compares overall mAP@0.5, examines per-class AP, and generates the final Task 1 figures used in `CASE_ANALYSIS.md`.

## Methodology

Both YOLO models were evaluated on the full TechTrack logistics dataset containing 9,525 images, 36,721 labeled objects, and 20 object classes. Predictions were generated using a score threshold of 0.5 and an NMS threshold of 0.4. Evaluation used IoU@0.5 and 11-point interpolated mAP through the implemented `metrics.py` pipeline.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        expected = candidate / "analysis" / "outputs" / "task1_model_summary_full.csv"
        if expected.exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing analysis/outputs/task1_model_summary_full.csv")

ROOT = find_repo_root(Path.cwd())
OUT = ROOT / "analysis" / "outputs"
FIG = ROOT / "analysis" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

summary_path = OUT / "task1_model_summary_full.csv"
comparison_path = OUT / "task1_per_class_ap_comparison_full.csv"
per_class_path = OUT / "task1_per_class_metrics_full.csv"

summary = pd.read_csv(summary_path)
comparison = pd.read_csv(comparison_path)
per_class = pd.read_csv(per_class_path)

ROOT, OUT, FIG

## Table 1: Overall Model Comparison Metrics

This table summarizes the aggregate performance of both candidate YOLO models on the full TechTrack dataset.

In [ ]:
summary_display = summary.copy()

summary_display = summary_display.rename(columns={
    "model": "Model",
    "mAP@0.5_11_point": "mAP@0.5",
    "total_ground_truth": "Ground Truth Objects",
    "total_predictions_after_nms": "Predictions After NMS",
    "score_threshold": "Score Threshold",
    "nms_threshold": "NMS Threshold",
    "map_iou_threshold": "Evaluation IoU Threshold",
    "eval_type": "Evaluation Score Type"
})

summary_display["mAP@0.5"] = summary_display["mAP@0.5"].round(4)

summary_display[[
    "Model",
    "mAP@0.5",
    "Ground Truth Objects",
    "Predictions After NMS",
    "Score Threshold",
    "NMS Threshold",
    "Evaluation IoU Threshold",
    "Evaluation Score Type"
]]

**Interpretation.** Model 2 achieved higher mAP@0.5 than Model 1 while producing slightly more post-NMS detections. This indicates that Model 2 is the stronger candidate baseline. The higher detection count is still worth examining in later threshold analysis because additional detections can be useful true positives, but they can also increase false positives or duplicate detections depending on threshold settings.

## Per-Class AP Table

This table compares Model 1 and Model 2 AP for each object class and computes the AP difference between the two models. The difference is calculated as Model 2 AP minus Model 1 AP.

In [ ]:
comparison_display = comparison.copy()

for col in ["model1_ap", "model2_ap", "ap_difference_model2_minus_model1"]:
    comparison_display[col] = comparison_display[col].round(4)

comparison_display = comparison_display.rename(columns={
    "class_id": "Class ID",
    "class_name": "Class",
    "ground_truth_count": "Ground Truth Count",
    "model1_ap": "Model 1 AP",
    "model2_ap": "Model 2 AP",
    "ap_difference_model2_minus_model1": "AP Difference: Model 2 - Model 1",
    "better_model": "Higher-AP Model"
})

comparison_display.sort_values("AP Difference: Model 2 - Model 1", ascending=False)

## Figure 1: Per-Class AP by Model

This figure shows the absolute AP score for each model by class. It is the main Task 1 performance plot because it shows both class difficulty and model-to-model performance.

In [ ]:
plot_df = comparison.sort_values("model2_ap", ascending=True).copy()

y = np.arange(len(plot_df))
bar_height = 0.38

plt.figure(figsize=(11, 9))
plt.barh(y - bar_height / 2, plot_df["model1_ap"], height=bar_height, label="Model 1")
plt.barh(y + bar_height / 2, plot_df["model2_ap"], height=bar_height, label="Model 2")

plt.yticks(y, plot_df["class_name"])
plt.xlabel("AP@0.5, 11-point interpolated")
plt.ylabel("Class")
plt.title("Task 1: Per-class AP by model")
plt.legend()
plt.tight_layout()

figure1_path = FIG / "task1_per_class_ap_by_model.png"
plt.savefig(figure1_path, dpi=200, bbox_inches="tight")
plt.show()

figure1_path

**Interpretation.** Figure 1 shows that Model 2 has higher AP than Model 1 for every object class. Some classes, such as QR code, van, truck, barcode, and traffic light, have relatively strong AP values, while classes such as freight container, road sign, fire, and ladder remain weaker. This matters for system design because the selected model should not be judged only by aggregate mAP; class-level weakness identifies where monitoring, threshold tuning, or data augmentation may still be needed.

## Figure 2: Per-Class AP Difference

This figure shows the AP difference between Model 2 and Model 1 for each class. Positive values show classes where Model 2 exceeded Model 1.

In [ ]:
difference_df = comparison.sort_values("ap_difference_model2_minus_model1", ascending=True).copy()

plt.figure(figsize=(10, 8))
plt.barh(difference_df["class_name"], difference_df["ap_difference_model2_minus_model1"])
plt.xlabel("AP difference: Model 2 minus Model 1")
plt.ylabel("Class")
plt.title("Task 1: Per-class AP difference between Model 2 and Model 1")
plt.tight_layout()

figure2_path = FIG / "task1_per_class_ap_difference.png"
plt.savefig(figure2_path, dpi=200, bbox_inches="tight")
plt.show()

figure2_path

**Interpretation.** Figure 2 shows that Model 2 exceeds Model 1 AP for every class. The largest AP advantages occur for forklift, barcode, gloves, traffic light, smoke, truck, and van. Several of these classes are operationally or safety relevant in warehouse perception, so the class-level AP differences support selecting Model 2 for downstream analysis.

## Table 2: Largest Model 2 AP Advantages

This table isolates the largest AP margins where Model 2 exceeds Model 1. This will help in the final report to discuss the most meaningful class-level differences without overwhelming the reader with all 20 rows.

In [ ]:
top_advantages = comparison.copy()
top_advantages = top_advantages.sort_values("ap_difference_model2_minus_model1", ascending=False).head(7)

top_advantages_display = top_advantages.rename(columns={
    "class_name": "Class",
    "ground_truth_count": "Ground Truth Count",
    "model1_ap": "Model 1 AP",
    "model2_ap": "Model 2 AP",
    "ap_difference_model2_minus_model1": "Model 2 AP Advantage"
})[[
    "Class",
    "Ground Truth Count",
    "Model 1 AP",
    "Model 2 AP",
    "Model 2 AP Advantage"
]]

for col in ["Model 1 AP", "Model 2 AP", "Model 2 AP Advantage"]:
    top_advantages_display[col] = top_advantages_display[col].round(4)

top_advantages_display

## Task 1 Conclusion

Model 2 achieved higher overall mAP@0.5 than Model 1 and higher AP for every object class. Therefore, Model 2 is selected as the baseline detector for the remaining sampling, NMS threshold, augmentation, and hard-negative-mining analyses.

The main trade-off is that Model 2 produced more post-NMS detections than Model 1. That is not automatically bad because Model 2 also had higher mAP, but it means later threshold analysis should examine whether the additional detections create avoidable false positives or duplicate detections in deployment.

## Appendix A: Full Task 1 Pipeline Code

This is the code used to generate the saved Task 1 prediction files and metric tables. 

from pathlib import Path
import sys
import argparse
import json
import time

import cv2
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from techtrack.modules.inference.model import Detector
from techtrack.modules.inference.nms import NMS
from techtrack.modules.utils.metrics import (
    match_detections,
    calculate_precision_recall_curve,
    calculate_map_x_point_interpolated,
)

OUT = ROOT / "analysis" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

DATASET_INDEX = OUT / "dataset_index.csv"
CLASS_FILE = ROOT / "techtrack" / "storage" / "yolo_model_1" / "logistics.names"

SCORE_THRESHOLD = 0.5
NMS_THRESHOLD = 0.4
MAP_IOU_THRESHOLD = 0.5
EVAL_TYPE = "combined"

MODELS = {
    "model1": {
        "weights": ROOT / "techtrack/storage/yolo_model_1/yolov4-tiny-logistics_size_416_1.weights",
        "cfg": ROOT / "techtrack/storage/yolo_model_1/yolov4-tiny-logistics_size_416_1.cfg",
        "names": ROOT / "techtrack/storage/yolo_model_1/logistics.names",
    },
    "model2": {
        "weights": ROOT / "techtrack/storage/yolo_model_2/yolov4-tiny-logistics_size_416_2.weights",
        "cfg": ROOT / "techtrack/storage/yolo_model_2/yolov4-tiny-logistics_size_416_2.cfg",
        "names": ROOT / "techtrack/storage/yolo_model_2/logistics.names",
    },
}

PRED_COLUMNS = [
    "model",
    "image_file",
    "image_path",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "class_id",
    "class_name",
    "object_score",
    "predicted_class_score",
    "combined_confidence",
    "class_scores_json",
]

GT_COLUMNS = [
    "image_file",
    "image_path",
    "class_id",
    "class_name",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
]


def load_classes():
    return [line.strip() for line in CLASS_FILE.read_text().splitlines() if line.strip()]


def yolo_label_to_xywh(label_path: Path, image_w: int, image_h: int, classes):
    rows = []
    text = label_path.read_text().strip()

    if not text:
        return rows

    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        class_id = int(float(parts[0]))

        if class_id < 0 or class_id >= len(classes):
            continue

        x_center = float(parts[1]) * image_w
        y_center = float(parts[2]) * image_h
        box_w = float(parts[3]) * image_w
        box_h = float(parts[4]) * image_h

        x = x_center - box_w / 2
        y = y_center - box_h / 2

        rows.append({
            "class_id": class_id,
            "class_name": classes[class_id],
            "bbox_x": x,
            "bbox_y": y,
            "bbox_w": box_w,
            "bbox_h": box_h,
        })

    return rows


def build_ground_truth(idx, classes, suffix, force=False):
    gt_path = OUT / f"task1_ground_truth{suffix}.csv"

    if gt_path.exists() and not force:
        print(f"[SKIP] Ground truth already exists: {gt_path}")
        return pd.read_csv(gt_path)

    print("=" * 80)
    print("Building ground-truth table")
    print("=" * 80)

    rows = []

    for row_num, (_, row) in enumerate(idx.iterrows(), start=1):
        image_path = ROOT / row["image_path"]
        label_path = ROOT / row["label_path"]

        frame = cv2.imread(str(image_path))
        if frame is None:
            print(f"[WARN] Could not read image for GT conversion: {image_path}")
            continue

        image_h, image_w = frame.shape[:2]

        for gt in yolo_label_to_xywh(label_path, image_w, image_h, classes):
            rows.append({
                "image_file": row["image_file"],
                "image_path": row["image_path"],
                "class_id": gt["class_id"],
                "class_name": gt["class_name"],
                "bbox_x": gt["bbox_x"],
                "bbox_y": gt["bbox_y"],
                "bbox_w": gt["bbox_w"],
                "bbox_h": gt["bbox_h"],
            })

        if row_num % 1000 == 0:
            print(f"[GT] processed {row_num}/{len(idx)} images")

    gt_df = pd.DataFrame(rows, columns=GT_COLUMNS)
    gt_df.to_csv(gt_path, index=False)

    print(f"[WRITE] {gt_path} rows={len(gt_df)}")
    return gt_df


def run_inference_for_model(model_name, paths, idx, classes, suffix, force=False):
    pred_path = OUT / f"task1_{model_name}_predictions{suffix}.csv"

    if pred_path.exists() and not force:
        print(f"[SKIP] Predictions already exist: {pred_path}")
        return pd.read_csv(pred_path)

    print("=" * 80)
    print(f"Running inference: {model_name}")
    print("=" * 80)

    detector = Detector(
        str(paths["weights"]),
        str(paths["cfg"]),
        str(paths["names"]),
        score_threshold=SCORE_THRESHOLD,
    )

    nms = NMS(
        score_threshold=SCORE_THRESHOLD,
        nms_iou_threshold=NMS_THRESHOLD,
    )

    rows = []
    start = time.time()

    for row_num, (_, row) in enumerate(idx.iterrows(), start=1):
        image_path = ROOT / row["image_path"]
        frame = cv2.imread(str(image_path))

        if frame is None:
            print(f"[WARN] Could not read image: {image_path}")
            continue

        outputs = detector.predict(frame)
        bboxes, class_ids, scores, class_scores = detector.post_process(outputs)

        filtered_boxes, filtered_classes, filtered_scores, filtered_class_scores = nms.filter(
            bboxes,
            class_ids,
            scores,
            class_scores,
        )

        for det_idx, bbox in enumerate(filtered_boxes):
            class_id = int(filtered_classes[det_idx])
            object_score = float(filtered_scores[det_idx])

            score_vector = np.asarray(filtered_class_scores[det_idx], dtype=float).ravel()
            score_vector = [float(v) for v in score_vector]

            if 0 <= class_id < len(score_vector):
                predicted_class_score = float(score_vector[class_id])
            else:
                predicted_class_score = 0.0

            combined_confidence = object_score * predicted_class_score

            rows.append({
                "model": model_name,
                "image_file": row["image_file"],
                "image_path": row["image_path"],
                "bbox_x": float(bbox[0]),
                "bbox_y": float(bbox[1]),
                "bbox_w": float(bbox[2]),
                "bbox_h": float(bbox[3]),
                "class_id": class_id,
                "class_name": classes[class_id] if 0 <= class_id < len(classes) else "unknown",
                "object_score": object_score,
                "predicted_class_score": predicted_class_score,
                "combined_confidence": combined_confidence,
                "class_scores_json": json.dumps(score_vector),
            })

        if row_num % 500 == 0:
            elapsed = time.time() - start
            print(
                f"[{model_name}] processed {row_num}/{len(idx)} images | "
                f"detections={len(rows)} | elapsed={elapsed:.1f}s"
            )

    pred_df = pd.DataFrame(rows, columns=PRED_COLUMNS)
    pred_df.to_csv(pred_path, index=False)

    elapsed = time.time() - start
    print(f"[WRITE] {pred_path} rows={len(pred_df)}")
    print(f"[DONE] {model_name} elapsed seconds: {elapsed:.2f}")

    return pred_df


def build_metric_lists(idx, pred_df, gt_df):
    pred_groups = {k: v for k, v in pred_df.groupby("image_file")} if len(pred_df) else {}
    gt_groups = {k: v for k, v in gt_df.groupby("image_file")} if len(gt_df) else {}

    boxes = []
    pred_classes = []
    scores = []
    cls_scores = []

    gt_boxes = []
    gt_classes = []

    for _, row in idx.iterrows():
        image_file = row["image_file"]

        pred_group = pred_groups.get(image_file)
        if pred_group is None:
            boxes.append([])
            pred_classes.append([])
            scores.append([])
            cls_scores.append([])
        else:
            boxes.append(pred_group[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].values.tolist())
            pred_classes.append(pred_group["class_id"].astype(int).tolist())
            scores.append(pred_group["object_score"].astype(float).tolist())
            cls_scores.append([
                json.loads(value)
                for value in pred_group["class_scores_json"].tolist()
            ])

        gt_group = gt_groups.get(image_file)
        if gt_group is None:
            gt_boxes.append([])
            gt_classes.append([])
        else:
            gt_boxes.append(gt_group[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].values.tolist())
            gt_classes.append(gt_group["class_id"].astype(int).tolist())

    return boxes, pred_classes, scores, cls_scores, gt_boxes, gt_classes


def evaluate_with_metrics_py(model_name, idx, pred_df, gt_df, classes):
    boxes, pred_classes, scores, cls_scores, gt_boxes, gt_classes = build_metric_lists(
        idx,
        pred_df,
        gt_df,
    )

    y_true, pred_scores = match_detections(
        boxes=boxes,
        classes=pred_classes,
        scores=scores,
        cls_scores=cls_scores,
        gt_boxes=gt_boxes,
        gt_classes=gt_classes,
        map_iou_threshold=MAP_IOU_THRESHOLD,
        eval_type=EVAL_TYPE,
    )

    precision, recall, thresholds = calculate_precision_recall_curve(
        y_true,
        pred_scores,
        num_classes=len(classes),
    )

    precision_recall_points = {
        class_id: list(zip(recall[class_id], precision[class_id]))
        for class_id in range(len(classes))
    }

    map_score = calculate_map_x_point_interpolated(
        precision_recall_points,
        num_classes=len(classes),
        num_interpolated_points=11,
    )

    per_class_rows = []

    for class_id, class_name in enumerate(classes):
        # Per-class AP using the same mAP function by remapping this class to class 0.
        per_class_ap = calculate_map_x_point_interpolated(
            {0: precision_recall_points[class_id]},
            num_classes=1,
            num_interpolated_points=11,
        )

        gt_count = int((gt_df["class_id"] == class_id).sum())
        prediction_count = int((pred_df["class_id"] == class_id).sum()) if len(pred_df) else 0

        per_class_rows.append({
            "model": model_name,
            "class_id": class_id,
            "class_name": class_name,
            "ground_truth_count": gt_count,
            "prediction_count": prediction_count,
            "ap_11_point": per_class_ap,
        })

    per_class_df = pd.DataFrame(per_class_rows)

    summary = {
        "model": model_name,
        "mAP@0.5_11_point": map_score,
        "total_ground_truth": int(len(gt_df)),
        "total_predictions_after_nms": int(len(pred_df)),
        "evaluation_rows": int(len(y_true)),
        "score_threshold": SCORE_THRESHOLD,
        "nms_threshold": NMS_THRESHOLD,
        "map_iou_threshold": MAP_IOU_THRESHOLD,
        "eval_type": EVAL_TYPE,
    }

    return summary, per_class_df


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--max-images", type=int, default=None)
    parser.add_argument("--force", action="store_true")
    args = parser.parse_args()

    classes = load_classes()
    idx = pd.read_csv(DATASET_INDEX)

    if args.max_images is not None:
        idx = idx.head(args.max_images).copy()
        suffix = f"_first_{args.max_images}"
        print(f"[INFO] Running Task 1 on first {args.max_images} images.")
    else:
        suffix = "_full"
        print("[INFO] Running Task 1 on full dataset.")

    print(f"[INFO] Images selected: {len(idx)}")
    print(f"[INFO] Classes: {len(classes)}")

    gt_df = build_ground_truth(idx, classes, suffix=suffix, force=args.force)

    summaries = []
    per_class_all = []

    for model_name, paths in MODELS.items():
        pred_df = run_inference_for_model(
            model_name,
            paths,
            idx,
            classes,
            suffix=suffix,
            force=args.force,
        )

        summary, per_class_df = evaluate_with_metrics_py(
            model_name,
            idx,
            pred_df,
            gt_df,
            classes,
        )

        summaries.append(summary)
        per_class_all.append(per_class_df)

    summary_df = pd.DataFrame(summaries)
    per_class_df = pd.concat(per_class_all, axis=0)

    comparison = (
        per_class_df.pivot(
            index=["class_id", "class_name", "ground_truth_count"],
            columns="model",
            values="ap_11_point",
        )
        .reset_index()
        .rename(columns={"model1": "model1_ap", "model2": "model2_ap"})
    )

    comparison["ap_difference_model2_minus_model1"] = (
        comparison["model2_ap"] - comparison["model1_ap"]
    )

    comparison["better_model"] = np.where(
        comparison["ap_difference_model2_minus_model1"] > 0,
        "model2",
        np.where(
            comparison["ap_difference_model2_minus_model1"] < 0,
            "model1",
            "tie",
        ),
    )

    summary_path = OUT / f"task1_model_summary{suffix}.csv"
    per_class_path = OUT / f"task1_per_class_metrics{suffix}.csv"
    comparison_path = OUT / f"task1_per_class_ap_comparison{suffix}.csv"

    summary_df.to_csv(summary_path, index=False)
    per_class_df.to_csv(per_class_path, index=False)
    comparison.to_csv(comparison_path, index=False)

    print()
    print("TASK 1 MODEL SUMMARY")
    print(summary_df.to_string(index=False))

    print()
    print("PER-CLASS AP COMPARISON")
    print(comparison.sort_values("ap_difference_model2_minus_model1", ascending=False).to_string(index=False))

    print()
    print("[WRITE]", summary_path)
    print("[WRITE]", per_class_path)
    print("[WRITE]", comparison_path)


if __name__ == "__main__":
    main()